In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="IXzKrxKhYSUYRyd5OJaN")
project = rf.workspace("norah-alshowair").project("using-phone-while-driving-qcvye-tjjun")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

YOLO("yolov8n.pt").train(data=f"{dataset.location}/data.yaml", epochs=30, imgsz=640, batch=32)

In [ ]:
# from ultralytics import YOLO


model = YOLO('runs/detect/train/weights/best.pt')

# الآن شغّل التقييم
metrics = model.val()
print(f"mAP@50: {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")

In [ ]:
results = model.predict(source='/content/img_11658 (1).jpg', conf=0.25, save=True)
import matplotlib.pyplot as plt
import cv2

# رسم النتيجة وعرضها
res_plotted = results[0].plot()
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()



In [ ]:
from google.colab import files

# تنزيل ملف النموذج الأفضل تلقائياً إلى جهازك
files.download('runs/detect/train/weights/best.pt')

In [ ]:
import os

# المسار الرئيسي لنتايج YOLO
detect_path = 'runs/detect'

if os.path.exists(detect_path):
    print("folders")
    folders = os.listdir(detect_path)
    for folder in folders:
        print(f"- {folder}")
else:
    print("لم يتم إنشاء مجلد runs/detect بعد.")

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="IXzKrxKhYSUYRyd5OJaN")
project = rf.workspace("norah-alshowair").project("seatbelt-detection-di1xz-u2rhe")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
from ultralytics import YOLO

# 1. تعريف وتحميل النموذج الأساسي
seatbelt_model = YOLO('yolov8n.pt')

results = seatbelt_model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=25,
    project='runs/seatbelt',
    name='train'
)

In [ ]:
# تقييم دقة النموذج على الـ Validation
metrics = seatbelt_model.val()
print(f"Seatbelt mAP@50: {metrics.box.map50:.3f}")
print(f"Seatbelt mAP@50-95: {metrics.box.map:.3f}")

In [ ]:
from google.colab import files

# تنزيل نموذج الجوال


# تنزيل نموذج الحزام
files.download('/content/runs/detect/runs/seatbelt/train/weights/best.pt')

In [ ]:
qqimport cv2
import matplotlib.pyplot as plt

# إجراء التوقع على صورة تجريبية
results = seatbelt_model.predict('/content/img_64937.jpg', conf=0.35)

# عرض الصورة بالصناديق
res_plotted = results[0].plot()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
import os
import cv2
import pandas as pd
from datetime import datetime, timedelta
from ultralytics import YOLO

# 1. تحميل النماذج
PHONE_MODEL_PATH = '/content/runs/detect/train/weights/best.pt'
BELT_MODEL_PATH = '/content/runs/detect/runs/seatbelt/train/weights/best.pt'

phone_model = YOLO(PHONE_MODEL_PATH)
belt_model = YOLO(BELT_MODEL_PATH)

def process_trip_video(
    video_path,
    trip_id="TRIP-001",
    start_time_str="2026-08-22T03:14:00",
    confidence_threshold=0.30,
    save_annotated_video=True
):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found at: {video_path}")

    # إعداد قراءة الفيديو
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    start_time = datetime.fromisoformat(start_time_str)

    # إعداد حفظ الفيديو النهائي
    out_video = None
    annotated_video_path = f"/content/annotated_{trip_id}.mp4"
    if save_annotated_video:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_video = cv2.VideoWriter(annotated_video_path, fourcc, fps, (width, height))

    violations_log = []

    # متغيرات تتبع مخالفة الجوال
    phone_in_violation = False
    phone_cooldown = 0
    phone_cooldown_limit = int(fps * 5)  # فاصل 5 ثوانٍ عند استمرار استخدام الجوال

    # متغيرات تتبع مخالفة عدم ارتداء الحزام (تسجيل مرة واحدة طالما لم يُربط)
    belt_logged_once = False
    unbuckled_consecutive_frames = 0
    belt_trigger_threshold = int(fps * 2)  # التنبيه بعد ثانيتين متواصلتين بدون حزام

    frame_idx = 0

    print(f"Processing video ({total_frames} frames) at {fps:.1f} FPS...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 2. تشغيل التوقع لكلا النموذجين
        phone_results = phone_model.predict(source=frame, conf=confidence_threshold, verbose=False)
        belt_results = belt_model.predict(source=frame, conf=confidence_threshold, verbose=False)

        phone_classes = [phone_model.names[int(c)] for c in phone_results[0].boxes.cls]
        belt_classes = [belt_model.names[int(c)] for c in belt_results[0].boxes.cls]

        # احتساب التوقيت الفعلي للإطار
        current_timestamp = start_time + timedelta(seconds=(frame_idx / fps))
        timestamp_str = current_timestamp.strftime("%Y-%m-%dT%H:%M:%S")

        # 3. منطق كشف استخدام الجوال
        if 'phone' in phone_classes:
            if not phone_in_violation and phone_cooldown <= 0:
                violations_log.append({
                    'trip_id': trip_id,
                    'violation_type': 'phone_use',
                    'timestamp': timestamp_str
                })
                phone_in_violation = True
                phone_cooldown = phone_cooldown_limit
        else:
            phone_in_violation = False

        if phone_cooldown > 0:
            phone_cooldown -= 1

        # 4. منطق كشف عدم ربط الحزام (تسجيل مرة واحدة فقط حتى يُربط مجدداً)
        has_seatbelt = any('seatbelt' in c.lower() for c in belt_classes)

        if not has_seatbelt:
            unbuckled_consecutive_frames += 1
            if unbuckled_consecutive_frames >= belt_trigger_threshold and not belt_logged_once:
                violations_log.append({
                    'trip_id': trip_id,
                    'violation_type': 'no_seatbelt',
                    'timestamp': timestamp_str
                })
                belt_logged_once = True
        else:
            unbuckled_consecutive_frames = 0
            belt_logged_once = False  # يُعاد التفعيل فقط إذا ربط الحزام ثم فكه مجدداً

        # 5. دمج ورسم الصناديق على الفيديو
        if save_annotated_video and out_video is not None:
            annotated_frame = phone_results[0].plot()
            annotated_frame = belt_results[0].plot(img=annotated_frame)
            out_video.write(annotated_frame)

        frame_idx += 1

    # إغلاق الموارد
    cap.release()
    if out_video is not None:
        out_video.release()

    # تصدير البيانات إلى CSV
    df = pd.DataFrame(violations_log)
    csv_path = f"/content/violations_{trip_id}.csv"
    df.to_csv(csv_path, index=False)

    print(f"Completed analysis for trip: {trip_id}")
    print(f"CSV file saved at: {csv_path}")
    if save_annotated_video:
        print(f"Annotated video saved at: {annotated_video_path}")

    return df

# تشغيل المعالجة
df_violations = process_trip_video(
    video_path='/content/7362606-hd_1920_1080_24fps (1).mp4',
    trip_id='TRIP-001',
    start_time_str='2026-08-22T03:14:00',
    confidence_threshold=0.30
)

print(df_violations)

In [ ]:
import glob

all_weights = glob.glob('/content/runs/**/best.pt', recursive=True)
print("الملفات الموجودة فعلياً:")
for idx, path in enumerate(all_weights):
    print(f"[{idx}] {path}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')